# Reelle datasett med JSON

I 1.9 hentet vi åpne data fra Felles datakatalog og SSB. Nå ser vi hva Europa har å by på, og jobber med data i JSON-format.

> Kjør cellene i rekkefølge med **Shift + Enter**. Cellene som henter data fra nettet, krever internett og pakken `pyjstat` (`pip install pyjstat`). Datakildene kan ha endret adresse siden dette ble skrevet.

## 1 Den offisielle portalen for europeiske data

[data.europa.eu](https://data.europa.eu) har over en million åpne datasett. Der rangeres også landene etter hvor langt de har kommet med åpne data.

Ett eksempel er *UDP – Internet speed at tourism destinations*. Datasettet er i **JSON-stat**-format: et JSON-format for statistikk med flere dimensjoner, som brukes av de nordiske statistikkbyråene, Eurostat og data.europa.eu. Biblioteket **pyjstat** leser det, og gjør det om til en pandas `DataFrame`:

* `pyjstat.Dataset.read(url)` leser datasettet
* `dataset.write('dataframe')` gjør det om til en DataFrame

In [ ]:
from pyjstat import pyjstat
import seaborn as sns
import matplotlib.pyplot as plt

url = ('https://urban.jrc.ec.europa.eu/api/udp/v2/en/data/?'
       'databrick_id=730&nutslevel=0&ts=TOURISM&nutsversion=-1&mpx=1')
dataset = pyjstat.Dataset.read(url)

df = dataset.write('dataframe')
print(df)

# Behold bare land og verdi, og gi kolonnene nye navn
df = df[['NUTS_CODE', 'value']]
df.columns = ['Land', 'Mbps']

# De ti med høyest hastighet
topp_10 = df.sort_values('Mbps', ascending=False).head(10)

sns.barplot(data=topp_10, x='Land', y='Mbps')
plt.xticks(rotation=30)
plt.title('Turistdestinasjoner i Europa med høyest internetthastighet')
plt.tight_layout()
plt.show()

Vi kvitter oss med kolonnene vi ikke trenger, gir de andre nye navn, sorterer og tar med de ti beste.

## 2 Eurostat

Vi vil lage et **sektordiagram** (kakediagram) over størrelsesforholdet mellom de fire største økonomiene i EU. På [Eurostat](https://ec.europa.eu/eurostat) finner vi *Gross domestic product at market prices*.

I databrowseren velger vi Tyskland, Frankrike, Italia og Spania for 2021, og *Current prices, million euro*. Under *Download* velger vi *JSON-stat*, *Data on this page only* og *Copy link*.

* `df.iloc[:, [3, 5]]` velger kolonnene vi trenger (land og BNP) etter posisjon – se [DataFrame.iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html).
* seaborn har ikke kakediagram, så vi bruker [`plt.pie`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.pie.html) fra matplotlib.

In [ ]:
from pyjstat import pyjstat
import matplotlib.pyplot as plt

url = ('https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/'
       'nama_10_gdp?format=JSON&unit=CP_MEUR&na_item=B1GQ&time=2021'
       '&geo=DE&geo=ES&geo=FR&geo=IT&lang=en')
df = pyjstat.Dataset.read(url).write('dataframe')
df = df.iloc[:, [3, 5]]
df.columns = ['Land', 'BNP']
print(df)

etiketter = ['Tyskland', 'Spania', 'Frankrike', 'Italia']   # samme rekkefølge som i tabellen!
plt.suptitle('Størrelsesforhold mellom de 4 største økonomiene i EU')
plt.title('Bruttonasjonalprodukt 2021')
plt.pie(df['BNP'], labels=etiketter, autopct='%1.0f%%')
plt.show()

## 3 Diagrammer over sykkelturer

*Fra Udirs eksempeloppgave for IT2 våren 2023, oppgave 14.* Vi tar med det første diagrammet her og de to andre i oppgavene.

> Du skal lage et program som leser inn informasjon fra datasettet og presenterer dette i tre diagrammer. Diagrammet skal vise de tre mest populære startlokasjonene og de tre minst populære startlokasjonene.

Datasettet er turene med Oslo Bysykkel i mai 2022. JSON-filen er på over 80 MB og CSV-filen over 30 MB – for store til å utforske i VS Code. CSV-filen kan åpnes i Excel (*Data → Fra tekst/CSV*), og der kan vi se at det er over 160 000 rader. Med en pivottabell finner vi mest og minst populære startlokasjoner.

Vi går over til Python. Cellen under prøver å laste ned filen. Uten nett lages et lite **tilfeldig generert** eksempelsett med samme kolonner, slik at resten av koden kan kjøres.

In [ ]:
import pandas as pd
import numpy as np

url = 'https://data.urbansharing.com/oslobysykkel.no/trips/v1/2022/05.json'

try:
    df = pd.read_json(url)           # eller: with open('05.json') as fil: df = pd.read_json(fil)
    print('Lastet ned', len(df), 'turer')
except Exception as feil:
    print('Fikk ikke lastet ned datasettet – lager eksempeldata.')
    rng = np.random.default_rng(1)
    stasjoner = ['Alexander Kiellands plass', 'Ringnes Park', 'Aker Brygge', 'Jernbanetorget',
                 'Bislett', 'Schous plass', 'Majorstuen', 'Gaustad T-bane', 'Kværnerveien',
                 'Tordenskiolds gate']
    vekter = np.array([30, 27, 26, 18, 15, 14, 12, 1, 1, 0.8]); vekter /= vekter.sum()
    n = 5000
    start = pd.Timestamp('2022-05-01') + pd.to_timedelta(rng.integers(0, 31*24*3600, n), unit='s')
    varighet = rng.integers(120, 3000, n)
    df = pd.DataFrame({
        'started_at': start,
        'ended_at': start + pd.to_timedelta(varighet, unit='s'),
        'duration': varighet,
        'start_station_name': rng.choice(stasjoner, n, p=vekter),
        'end_station_name': rng.choice(stasjoner, n, p=vekter),
    })

print(df.columns.tolist())
df.head()

[`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) på kolonnen med startlokasjoner gir en sortert **Series** med antall forekomster av hver stasjon. `head(3)` gir de tre første og `tail(3)` de tre siste, og `pd.concat` skjøter dem sammen.

En `DataFrame` er en todimensjonal tabell. En `Series` er endimensjonal, og indeksert med navn (her stasjonsnavnet):

In [ ]:
s = df['start_station_name'].value_counts()
mest_pop = s.head(3)
minst_pop = s.tail(3)
data = pd.concat([mest_pop, minst_pop])

print(data.iloc[2])              # etter posisjon
print(data[data.index[2]])       # etter navn
print(data.index[2])

Nå kan vi bruke `data.index` langs x-aksen. `ax` er et **Axes**-objekt som inneholder det meste av diagrammet. Resten er formatering: tittel, verdier over stolpene (`bar_label`), aksetitler og skråstilte etiketter. `margins` gir plass til verdiene over stolpene, og `tight_layout` gir plass til etikettene.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ax = sns.barplot(x=data.index, y=data.values)
ax.set(title='Tre mest og minst populære startlokasjoner',
       xlabel='Startlokasjon', ylabel='Antall (popularitet)')
ax.bar_label(ax.containers[0])
plt.xticks(rotation=30, ha='right')
plt.margins(0.05, 0.1)
plt.tight_layout()
plt.show()

## 4 Oppgaver

Vi fortsetter med [eksempeloppgaven fra Udir](https://www.udir.no/eksamen-og-prover/eksamen/) for våren 2023, oppgave 14, med datasettet over.

### Oppgave 1
Lag et diagram som viser de tre **sluttlokasjonene** med høyest **gjennomsnittlig varighet** på turer til lokasjonen. Tips: `groupby` og `mean`.

In [ ]:
# Oppgave 1

### Oppgave 2
Lag et diagram som viser **antall turer totalt per ukedag**. Tips: `pd.to_datetime(...)` og `.dt.day_name()` eller `.dt.dayofweek`.

In [ ]:
# Oppgave 2